get data

transform clean data

prepare pipeline

compare 5 algo

propose to search hyper parameters for 2 algos



In [16]:
import pandas as pd  
# data = pd.read_csv("data/2025-03-13_data_production.csv",index_col=0)
data = pd.read_csv("data/new_data.csv",index_col=0)
y = pd.read_csv("data/y.csv",index_col=0)['QC_Conc_XY'].astype(float)

In [17]:
# Handeling duplicates
duplicates = data.duplicated(subset=['Batch_XY_name','Batch_XY_date'])
data = data.loc[~duplicates]

# Handeling null values
null_values = list()
for col in data.columns:
    if data[col].isnull().sum() > data.shape[0]/3:
        null_values.append(col)
data.drop(columns=null_values,inplace=True)

# Handeling unique values
variable_to_remove = list()
count_unique = data.astype(str).describe().T

# find columns names that have unique values or identical values (not usefull for ML model)
for i in range(count_unique.shape[0]):
    if count_unique.iloc[i,1] == 280 or count_unique.iloc[i,1] == 1:
        variable_to_remove.append(data.columns[i])
data[variable_to_remove].astype(str).describe()
data.drop(columns=variable_to_remove,inplace=True)


# Data Engineering
data['conc_XX'] = data['Batch_XY_XX_masse'] / data['Batch_XY_YY_Volume']
data.drop(columns=['Batch_XY_XX_masse','Batch_XY_YY_Volume'],inplace=True)


# Handeling time data (transform to datetime format)
datetime_to_remove = list()
for col in data.columns:
    if 'heure' in col or 'date' in col:
        data[col] = pd.to_datetime(data[col])
        datetime_to_remove.append(col)

data['month_production'] = data['Batch_XY_date'].dt.month
data['day_production_start'] = data['Batch_XY_date'].dt.day
data['days_exfo'] = (data['Batch_XY_heure_fin'] - data['Batch_XY_heure_debut']).dt.days



In [18]:
# Ajoute data autre source
import joblib
import json
import requests

def get_maree_data(years : list):
    """Cette fonction se connecte à une API externe pour récuperer des données si elle ne sont pas déja présentes"""
    try :
        gde_marees = joblib.load("data/gde_maree.bin")
        print("Donées grandes marées disponibles")
        return gde_marees

    except:
        print("Téléchargement données grandes marées")
        gde_marees = pd.DataFrame()
        for year in years:
            url = f"https://data.stmalo-agglomeration.fr/api/explore/v2.1/catalog/datasets/grandes-marees-a-saint-malo/records?limit=20&refine=date%3A%22{year}%22"
            gde_marees = pd.concat([gde_marees,pd.DataFrame(json.loads(requests.get(url).content)['results'])],axis=0)
        # gde_marees
        gde_marees["date"] = pd.to_datetime(gde_marees["date"])
        joblib.dump(gde_marees,"data/gde_maree.bin")
        return gde_marees
    
def check_if_within_range(row, check_times):
    """
    Compte le nombre de datetimes dans check_times qui tombent dans la plage
    définie par Heure_debut et Heure_ajout2.
    """
    return sum(row["Batch_XY_heure_debut"] <= check_time <= row["Batch_XY_heure_fin"] for check_time in check_times)


gde_marees = get_maree_data(years = [2022,2023,2024,2025])
data["exfo_gde_maree"] =  data.apply(lambda row: check_if_within_range(row, gde_marees['date']), axis=1)
data.drop(columns=datetime_to_remove, inplace=True)


Donées grandes marées disponibles


In [19]:
# data.select_dtypes('number').plot()
# data.drop(columns=['days_exfo']).select_dtypes('number').plot()
# data.drop(columns=['days_exfo','Batch_XY_Agitation']).select_dtypes('number').plot()
# data[['month_production','exfo_gde_maree']].plot()
# data.select_dtypes('number').boxplot()
# import matplotlib.pyplot as plt
# plt.show()

days exfo ont des valeurs absurdes remplacees par la moyenne

les valeurs negatives mise positives

on remplace les variables environementales nulles par la moyenne des temperature pour un même mois

In [20]:
# impute valeur absurdes
import numpy as np

for var in ['Batch_XY_Temperature','Batch_XY_room_HR','Batch_XY_room_T','conc_XX','days_exfo','exfo_gde_maree']:
    _ = data.loc[data[var] != 0, ["month_production",var]]
    _dict = _.groupby(['month_production'])[var].mean().to_dict()

    if not var in ['days_exfo','exfo_gde_maree']:
        data.loc[data[var] == 0, var] = data.loc[data[var] == 0, 'month_production'].map(_dict)
    else:
        Q1 = np.percentile(data[var], 25)
        Q3 = np.percentile(data[var], 75)
        IQR = Q3 - Q1
        # Définir les seuils pour les outliers
        low_fly = Q1 - 1.5 * IQR
        up_fly = Q3 + 1.5 * IQR
        data.loc[(data[var] < low_fly) | (data[var]> up_fly),var] = data.loc[(data[var] < low_fly) | (data[var]> up_fly), 'month_production'].map(_dict)


/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_66705/3561065607.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[103.21621622  -1.75         3.35294118   3.35294118 370.54545455
   5.64        22.9047619   22.9047619  103.21621622   4.23076923
 -39.6875     -39.6875     -39.6875     -39.6875    ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[(data[var] < low_fly) | (data[var]> up_fly),var] = data.loc[(data[var] < low_fly) | (data[var]> up_fly), 'month_production'].map(_dict)
/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_66705/3561065607.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 2.66666667  2.66666667  2.66666667  4.5         4.5         4.5
  6.          6.          6.          1.66666667  1.66666667  1.
  1.57142857  1.57142857

In [21]:
data.columns

Index(['Batch_OGD_Technicien', 'Batch_OGD_KC8_batch', 'Batch_OGD_Temperature',
       'Batch_OGD_Agitation', 'Batch_OGD_room_HR', 'Batch_OGD_room_T',
       'Batch_OGD_Analyses', 'conc_KC8', 'month_production',
       'day_production_start', 'days_exfo', 'exfo_gde_maree'],
      dtype='object')

In [22]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

categorical_columns = ['Batch_XY_Technicien','Batch_XY_XX_batch', 'Batch_XY_Analyses',
                        'month_production','day_production_start','exfo_gde_maree']

numerical_columns = ['Batch_XY_Temperature','Batch_XY_Agitation','Batch_XY_room_HR',
                    'conc_XX','Batch_XY_room_T','days_exfo']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(),numerical_columns),
        ('categorical',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),categorical_columns)])

preprocessor.fit(data)

ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                 ['Batch_OGD_Temperature',
                                  'Batch_OGD_Agitation', 'Batch_OGD_room_HR',
                                  'conc_KC8', 'Batch_OGD_room_T',
                                  'days_exfo']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Batch_OGD_Technicien', 'Batch_OGD_KC8_batch',
                                  'Batch_OGD_Analyses', 'month_production',
                                  'day_production_start', 'exfo_gde_maree'])])

In [23]:
preprocessor.fit_transform(X).shape

(280, 123)

In [24]:
# configure mlflow
import mlflow
from datetime import datetime
mlflow.set_tracking_uri('http://127.0.0.1:8080')
date = datetime.now().strftime('%Y-%m-%d')
mlflow.set_experiment(f"/{date}Carbon_waters_scan_models_newX")


<Experiment: artifact_location='file:///Users/remicazelles/Documents/Travail/2023-SImplon_microsoft/PROJET_FINAL/ML_supervized/artifcat_RC/109156863891557588', creation_time=1741898619600, experiment_id='109156863891557588', last_update_time=1741898619600, lifecycle_stage='active', name='/2025-03-13Carbon_waters_scan_models_newX', tags={}>

In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression,ElasticNet,SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, ShuffleSplit

# prepare search parameters
test_sizes = [0.1,0.2,0.3,0.4,0.5]
dummy = DummyRegressor()
linear = LinearRegression()
elastic = ElasticNet(alpha=0.1, l1_ratio=0.8, max_iter=5000)
sgd = SGDRegressor(max_iter=10000, tol=1e-3, loss='epsilon_insensitive',penalty='elasticnet')
forest = RandomForestRegressor(max_depth=10,max_features='log2',min_samples_split=20)
models = [dummy, linear,elastic,sgd,forest]

In [26]:
X=data.copy()
all_results = pd.DataFrame()
# run experiment
for experiment_set in product(test_sizes,models):
    with mlflow.start_run(nested=True):

        # perfom tests
        test_size, model = experiment_set
        pipeline = Pipeline([('preprocessor',preprocessor),
                                ('regressor', model)])
        
        cv = cross_validate(pipeline,X,y,cv=ShuffleSplit(n_splits=5,test_size=test_size),return_train_score=True)
        
        # save result in dataframe
        results = pd.DataFrame(cv)
        if results['test_score'].mean() <= 0:
            results['test_score'] = 0
        all_results = pd.concat([all_results,results.T.mean(axis=1)],axis=1)

        # log data in mlflow
        mlflow.log_params({"model":model,
                "test_size":test_size})
        mlflow.log_metrics({'score_train_mean':results['train_score'].mean(),
                            'score_train_std':results['train_score'].std(),
                            'score_test_mean':results['test_score'].mean(),
                            'score_test_std':results['test_score'].std()})

print("RESULTS:\n")
all_results.columns = [a for a in product(test_sizes,models)]
print(all_results.T.sort_values(by=['test_score','train_score'], ascending=[False,False]))
# display best models
# best_5_models = results.sort_values(by=['score_test_mean','score_test_std','score_train_mean','score_train_std'],
#                     ascending=[False,True,False,True]).iloc[:5,:].values
# print(best_5_models)    



🏃 View run persistent-snail-323 at: http://127.0.0.1:8080/#/experiments/109156863891557588/runs/14d065936c464e19a79e466a017d090c
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/109156863891557588
🏃 View run brawny-hound-57 at: http://127.0.0.1:8080/#/experiments/109156863891557588/runs/1f18ed3441e94eca8091abac1f47c526
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/109156863891557588
🏃 View run flawless-sloth-270 at: http://127.0.0.1:8080/#/experiments/109156863891557588/runs/e9934173b7a74575bedbdd089d282fdc
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/109156863891557588
🏃 View run spiffy-bird-685 at: http://127.0.0.1:8080/#/experiments/109156863891557588/runs/7010032db4bf4c6e909d43cdc87a39f5
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/109156863891557588
🏃 View run invincible-stag-877 at: http://127.0.0.1:8080/#/experiments/109156863891557588/runs/fbde2743498640cd89233601f4d7ca2f
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/10